# Feature engineering, round 2 — and a control that tests round 1's instrument

`feature_engineering.ipynb` produced the project's first positive result: 31 engineered columns moved
the Cover-Hart bracket from $[0.2135, 0.3757]$ to $[0.2053, 0.3632]$ (McNemar $p = 9.3\times10^{-6}$)
and the single $\alpha{=}0.25$ score from 0.6157 to 0.6258. `NEXT_STEPS.md` then made further feature
work the top priority, since it is the **only lever in the project that has ever moved the bound
rather than approached it**.

This notebook does two things.

## 1. Round-2 features (the extension)

Three of round 1's directions are unavailable or exhausted:

| Direction from `NEXT_STEPS.md` | Status |
|---|---|
| richer flow statistics (percentiles) | **impossible** - requires re-running CICFlowMeter on the PCAPs, which we do not have |
| host / temporal context | **impossible on this file** - `CIC-UNSW-Data.csv` has **no IP, port or timestamp column**, only the 76 flow statistics |
| more algebra on existing columns | **available** - this notebook |

So round 2 is 38 further columns, chosen to attack the specific structures the overlap analysis
named, rather than generically:

- **response structure** - `no_bwd_pkt`, `no_bwd_bytes`, `act_data_frac_fwd`, `bwd_pkt_share`.
  A reconnaissance scan is a flow that *gets no answer*; Exploits get answers. Exploits<->Recon is the
  largest confusion pair (451) with **zero** exact ambiguity, so it must be a geometric fix.
- **control-plane share** - `ctrl_share`, `handshake_bal`, `r_ack_psh`, `rst_flag`. DoS<->Exploits
  (442 confusions, zero ambiguity) differ in how flows *terminate*, not in how much they carry.
- **bounded re-expressions** - `pktlen_min_rel`, `fwd_len_range_norm`, `bwd_pkt_share`,
  `fwd_rate_share`. Round 1 used unbounded ratios $a/b$; a ratio that can reach $10^6$ dominates a
  Euclidean neighbourhood on its own. Bounded forms of the *same* quantity change the geometry the
  k-NN diagnostic and the DEC-space combiner both depend on.
- **time-share and dispersion** - IAT shares of duration, active/idle coefficients of variation.
- **subflow, bulk and window structure** - untouched in round 1.

## 2. A CONTROL that tests round 1's own instrument

The primary diagnostic in this project is the 1-NN error, measured in **RobustScaler Euclidean
space**. That number is a property of the *metric*, not only of the information available:

> A strictly monotone per-column transform (e.g. $\operatorname{sign}(x)\log(1+|x|)$) adds **zero**
> information. A tree is invariant to it. But it changes every distance, so it can move $e_{1NN}$ and
> therefore the Cover-Hart bracket.

If a signed-log rescaling of the *existing* columns moves the bracket as much as round 1's new
columns did, then round 1's headline was measuring geometry, not information. That would not overturn
the score gain (+0.0101, corroborated by purity), but it would materially weaken how the bracket
result should be stated in the thesis.

**This is the check that should have accompanied round 1, and it is run here before any round-2 claim
is made.**

| Arm | Contents | Information vs `DERIVED` | Tree-visible change |
|---|---|---|---|
| `DERIVED` | round-1 space (75) | - | - |
| `D2` | `DERIVED` + 38 round-2 columns | **more** | yes |
| `DERIVED_log` | `DERIVED`, signed-log per column | **identical** | no (monotone) |
| `D2_log` | `D2`, signed-log per column | same as `D2` | no |

Reading the 2x2: a bracket move along the `log` axis is **pure metric**; a move along the `D2` axis at
fixed metric is **information**. The e2e score along the `log` axis should be ~0 and is measured, not
assumed.

> **Runtime ~15 min.** 1-NN diagnostics are seconds each; the four matched
> gate + $\alpha{=}0.25$ Stage-B fits dominate.

## 1. Setup, splits and the round-1 spaces (reproduced exactly)

In [1]:
import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.stats import binomtest
from xgboost import XGBClassifier

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS, EPS = 42, 1e-6

Xraw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
yraw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([Xraw, yraw], axis=1).drop_duplicates().reset_index(drop=True)
Xraw, y = df[Xraw.columns], df["Label"]

nun  = Xraw.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)
corr = FULL.corr(); upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red  = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)

# no host / temporal context is recoverable: confirm the file has no identifier columns.
# whole-token match, so "Idle Mean" is not counted as an "id" column.
import re as _re
_TOK = {"ip", "ips", "port", "ports", "sport", "dport", "timestamp", "time", "date",
        "src", "dst", "source", "destination", "protocol", "proto", "id", "flow_id"}
ident = [c for c in Xraw.columns if _TOK & set(_re.split(r"[^a-z]+", c.lower()))]
print("identifier / timestamp columns present:", ident if ident else
      "NONE -> per-host and per-window aggregates are impossible on this file")
print("rows after dedup %d | FULL %d | BASE %d" % (len(df), FULL.shape[1], BASE.shape[1]))

identifier / timestamp columns present: NONE -> per-host and per-window aggregates are impossible on this file
rows after dedup 306173 | FULL 67 | BASE 44


In [2]:
def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

def build_round1(F):
    """Verbatim from feature_engineering.ipynb - the incumbent 31 columns."""
    d = {}
    g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
    fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
    fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
    tot_pk, tot_by = fpk + bpk, fby + bby
    d["r_pkt_fwd_bwd"] = safe(fpk, bpk);   d["r_byte_fwd_bwd"] = safe(fby, bby)
    d["asym_pkt"] = safe(fpk - bpk, tot_pk); d["asym_byte"] = safe(fby - bby, tot_by)
    d["bpp_fwd"] = safe(fby, fpk); d["bpp_bwd"] = safe(bby, bpk); d["bpp_flow"] = safe(tot_by, tot_pk)
    d["r_bpp"] = safe(d["bpp_fwd"], d["bpp_bwd"])
    d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
    d["cv_fwd_iat"]  = safe(g("Fwd IAT Std"),  g("Fwd IAT Mean"))
    d["cv_bwd_iat"]  = safe(g("Bwd IAT Std"),  g("Bwd IAT Mean"))
    d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))
    d["cv_fwd_len"] = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
    d["cv_bwd_len"] = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
    d["cv_pkt_len"] = safe(g("Packet Length Std"), g("Packet Length Mean"))
    d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"),
                                 g("Fwd Packet Length Mean"))
    d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"),
                                 g("Bwd Packet Length Mean"))
    for flag in ("FIN", "SYN", "RST", "PSH", "ACK"):
        d["frac_" + flag] = safe(g(flag + " Flag Count"), tot_pk)
    d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))
    d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
    d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
    d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)
    dur = g("Flow Duration")
    d["active_share"] = safe(g("Active Mean"), dur); d["idle_share"] = safe(g("Idle Mean"), dur)
    d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean")); d["dur_per_pkt"] = safe(dur, tot_pk)
    d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))
    D = pd.DataFrame(d, index=F.index).astype(np.float64)
    return D.replace([np.inf, -np.inf], 0.0).fillna(0.0)

def build_round2(F):
    """38 new columns. Grouped by the confusion structure each group targets."""
    d = {}
    g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
    fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
    fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
    tot_pk, tot_by = fpk + bpk, fby + bby
    dur = g("Flow Duration")

    # --- response structure: Exploits <-> Reconnaissance (451 confusions, zero exact ambiguity) ---
    # a scan is a flow that gets no answer; an exploit gets one.
    d["no_bwd_pkt"]       = (bpk == 0).astype(float)
    d["no_bwd_bytes"]     = (bby == 0).astype(float)
    d["tiny_flow"]        = (tot_pk <= 3).astype(float)
    d["zero_dur"]         = (dur == 0).astype(float)
    d["act_data_frac_fwd"] = safe(g("Fwd Act Data Pkts"), fpk)
    d["bwd_pkt_share"]    = safe(bpk, tot_pk)          # bounded twin of Down/Up Ratio
    d["bwd_byte_share"]   = safe(bby, tot_by)

    # --- control plane: DoS <-> Exploits (442 confusions, zero exact ambiguity) ---
    # same volume, different termination.
    syn, fin, rst = g("SYN Flag Count"), g("FIN Flag Count"), g("RST Flag Count")
    d["ctrl_share"]   = safe(syn + fin + rst, tot_pk)
    d["handshake_bal"] = safe(syn - fin, syn + fin)     # in [-1, 1]
    d["r_ack_psh"]    = safe(g("ACK Flag Count"), g("PSH Flag Count"))
    d["rst_flag"]     = (rst > 0).astype(float)
    d["frac_fwd_psh"] = safe(g("Fwd PSH Flags"), fpk)

    # --- time structure ---
    d["fwd_iat_share"]   = safe(g("Fwd IAT Total"), dur)
    d["bwd_iat_share"]   = safe(g("Bwd IAT Total"), dur)
    d["iat_min_rel"]     = safe(g("Flow IAT Min"), g("Flow IAT Mean"))
    d["iat_max_rel"]     = safe(g("Flow IAT Max"), g("Flow IAT Mean"))
    d["fwd_iat_span_rel"] = safe(g("Fwd IAT Max") - g("Fwd IAT Min"), g("Fwd IAT Mean"))
    d["bwd_iat_span_rel"] = safe(g("Bwd IAT Max") - g("Bwd IAT Min"), g("Bwd IAT Mean"))
    d["cv_active"]       = safe(g("Active Std"), g("Active Mean"))
    d["cv_idle"]         = safe(g("Idle Std"), g("Idle Mean"))
    d["active_span_rel"] = safe(g("Active Max") - g("Active Min"), g("Active Mean"))
    d["idle_span_rel"]   = safe(g("Idle Max") - g("Idle Min"), g("Idle Mean"))
    d["dur_per_byte"]    = safe(dur, tot_by)

    # --- size structure, in BOUNDED form where round 1 used an unbounded ratio ---
    pl_mean = g("Packet Length Mean")
    d["pktlen_min_rel"] = safe(g("Packet Length Min"), pl_mean)
    d["pktlen_max_rel"] = safe(g("Packet Length Max"), pl_mean)
    d["avg_pkt_size_rel"] = safe(g("Average Packet Size"), pl_mean)
    d["seg_ratio"]     = safe(g("Fwd Segment Size Avg"), g("Bwd Segment Size Avg"))
    d["seg_min_rel"]   = safe(g("Fwd Seg Size Min"), g("Fwd Segment Size Avg"))
    d["hdr_share"]     = safe(g("Fwd Header Length") + g("Bwd Header Length"), tot_by)
    d["payload_share_bwd"] = safe(bby - g("Bwd Header Length"), bby)
    d["fwd_len_range_norm"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"),
                                   g("Fwd Packet Length Max"))   # in [0, 1]
    d["bwd_len_range_norm"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"),
                                   g("Bwd Packet Length Max"))

    # --- subflow / bulk / window structure: untouched by round 1 ---
    d["subflow_fwd_ratio"] = safe(g("Subflow Fwd Bytes"), fby)
    d["subflow_bwd_ratio"] = safe(g("Subflow Bwd Bytes"), bby)
    d["subflow_pkt_ratio"] = safe(g("Subflow Fwd Packets"), fpk)
    d["bulk_mean_size_bwd"] = safe(g("Bwd Bytes/Bulk Avg"), g("Bwd Packet/Bulk Avg"))
    d["bulk_share_bwd"]    = safe(g("Bwd Bytes/Bulk Avg"), bby)
    d["fwd_rate_share"]    = safe(g("Fwd Packets/s"), g("Fwd Packets/s") + g("Bwd Packets/s"))
    d["win_per_seg_fwd"]   = safe(g("FWD Init Win Bytes"), g("Fwd Segment Size Avg"))
    d["win_missing_bwd"]   = (g("Bwd Init Win Bytes") < 0).astype(float)

    D = pd.DataFrame(d, index=F.index).astype(np.float64)
    return D.replace([np.inf, -np.inf], 0.0).fillna(0.0)

def drop_const(D):
    n = D.nunique(); return D.drop(columns=n[n <= 1].index.tolist())

R1 = drop_const(build_round1(FULL))
R2 = drop_const(build_round2(FULL))
DERIVED = pd.concat([BASE, R1], axis=1)
D2      = pd.concat([BASE, R1, R2], axis=1)

def slog(D):
    """sign(x) * log1p(|x|) - strictly monotone per column, so ZERO information change."""
    return pd.DataFrame(np.sign(D.values) * np.log1p(np.abs(D.values)),
                        index=D.index, columns=D.columns)

SPACES = {"DERIVED": DERIVED, "D2": D2,
          "DERIVED_log": slog(DERIVED), "D2_log": slog(D2)}

print("round-1 columns kept : %d" % R1.shape[1])
print("round-2 columns kept : %d of 38  (dropped as constant: %s)"
      % (R2.shape[1], ", ".join(sorted(set(build_round2(FULL).columns) - set(R2.columns))) or "none"))
print()
for k, v in SPACES.items():
    print("  %-12s %3d features" % (k, v.shape[1]))

round-1 columns kept : 31
round-2 columns kept : 38 of 38  (dropped as constant: win_missing_bwd, zero_dur)

  DERIVED       75 features
  D2           113 features
  DERIVED_log   75 features
  D2_log       113 features


Splits are the seed-42 stratified 80/20 used by every notebook in the project, so the test set is
row-for-row the one every reported number uses.

In [3]:
idx_all = np.arange(len(y))
itr, ite = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
y_arr = y.values
att_tr, att_te = (y_arr[itr] != 0), (y_arr[ite] != 0)

attack_labels = sorted(l for l in np.unique(y_arr) if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels)
FAMN = [LABELS[idx_to_fam[i]] for i in range(NC)]
fam_tr = pd.Series(y_arr[itr][att_tr]).map(fam_to_idx).values
fam_te = pd.Series(y_arr[ite][att_te]).map(fam_to_idx).values
print("train+val %d (%d attack) | test %d (%d attack) | %d families"
      % (len(itr), att_tr.sum(), len(ite), att_te.sum(), NC))

train+val 244938 (64936 attack) | test 61235 (16234 attack) | 9 families


## 2. PRIMARY DIAGNOSTIC — 1-NN error and the Cover-Hart bracket, four arms

Neighbours of **test** attack rows among **train+val** attack rows, RobustScaler fitted on train only,
Stage-B scope (9 families) — identical protocol to `knn_class_overlap.ipynb` and round 1, so the
numbers are directly comparable to the published $[0.213, 0.376]$ and round 1's $[0.2053, 0.3632]$.

The per-row 1-NN correctness vector is retained for each arm so every comparison can be **paired**
(McNemar exact), the same test round 1 used.

In [4]:
def cover_hart(e1nn, c):
    inner = 1 - (c / (c - 1)) * e1nn
    return np.nan if inner < 0 else (c - 1) / c * (1 - np.sqrt(inner))

rows, HIT, NBR = [], {}, {}
for name, F in SPACES.items():
    Xa_tr, Xa_te = F.iloc[itr][att_tr], F.iloc[ite][att_te]
    sc = RobustScaler().fit(Xa_tr)
    t0 = time.time()
    nn = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(sc.transform(Xa_tr))
    _, ii = nn.kneighbors(sc.transform(Xa_te))
    hit = fam_tr[ii[:, 0]] == fam_te
    HIT[name] = hit
    e1 = float((~hit).mean()); lo = cover_hart(e1, NC)
    rows.append(dict(space=name, features=F.shape[1], e_1NN=round(e1, 4),
                     e_star_lo=round(lo, 4), e_star_hi=round(e1, 4),
                     acc_ceil_lo=round(1 - e1, 4), acc_ceil_hi=round(1 - lo, 4),
                     secs=round(time.time() - t0, 1)))
ch = pd.DataFrame(rows)
print(ch.to_string(index=False))
print("\nreference: BASE 44 feat -> e_1NN 0.3757 (published 0.3755); round-1 DERIVED -> 0.3629")

def mcnemar(a, b):
    """exact McNemar on paired 1-NN correctness: b = a-right/b-wrong, c = a-wrong/b-right"""
    nb = int((HIT[a] & ~HIT[b]).sum()); nc = int((~HIT[a] & HIT[b]).sum())
    p = binomtest(nb, nb + nc, 0.5).pvalue if nb + nc else 1.0
    return nb, nc, nc - nb, p

print("\npaired McNemar exact tests (net = rows the SECOND arm gets right that the first does not)")
pairs = [("DERIVED", "D2"), ("DERIVED", "DERIVED_log"), ("D2", "D2_log"),
         ("DERIVED_log", "D2_log"), ("DERIVED", "D2_log")]
mc = pd.DataFrame([dict(comparison="%s -> %s" % p_, b=b, c=c, net=n,
                        p=("%.3g" % pv), sig=("yes" if pv < 0.05 else "no"))
                   for p_ in pairs for b, c, n, pv in [mcnemar(*p_)]])
print(mc.to_string(index=False))

      space  features  e_1NN  e_star_lo  e_star_hi  acc_ceil_lo  acc_ceil_hi  secs
    DERIVED        75 0.3632     0.2053     0.3632       0.6368       0.7947   3.3
         D2       113 0.3517     0.1979     0.3517       0.6483       0.8021   1.3
DERIVED_log        75 0.3928     0.2248     0.3928       0.6072       0.7752   1.0
     D2_log       113 0.3878     0.2215     0.3878       0.6122       0.7785   1.1

reference: BASE 44 feat -> e_1NN 0.3757 (published 0.3755); round-1 DERIVED -> 0.3629

paired McNemar exact tests (net = rows the SECOND arm gets right that the first does not)
            comparison    b    c  net        p sig
         DERIVED -> D2  583  769  186 4.68e-07 yes
DERIVED -> DERIVED_log 1479  999 -480  4.8e-22 yes
          D2 -> D2_log 1584  998 -586 6.38e-31 yes
 DERIVED_log -> D2_log  673  753   80   0.0364 yes
     DERIVED -> D2_log 1662 1262 -400 1.47e-13 yes


### 2.1 Reading the 2x2 — how much of a bracket move is information?

The design isolates the two effects. `DERIVED -> D2` holds the metric fixed and adds columns
(information). `DERIVED -> DERIVED_log` holds the information fixed and changes the metric
(**pure geometry** — a tree cannot see this transform at all).

In [5]:
e = {r["space"]: r["e_1NN"] for _, r in ch.iterrows()}
info_effect  = e["D2"] - e["DERIVED"]
metric_effect = e["DERIVED_log"] - e["DERIVED"]
combined      = e["D2_log"] - e["DERIVED"]
print("effect on e_1NN (negative = tighter bracket)")
print("  information only (DERIVED -> D2)       : %+.4f" % info_effect)
print("  METRIC only     (DERIVED -> DERIVED_log): %+.4f   <- adds ZERO information" % metric_effect)
print("  both            (DERIVED -> D2_log)     : %+.4f" % combined)
print("\nround 1 for scale: BASE -> DERIVED was %+.4f (0.3757 -> 0.3629), p = 9.3e-06" % (0.3629 - 0.3757))
ratio = abs(metric_effect) / abs(0.3629 - 0.3757)
print("\nthe pure-metric move is %.0f%% the size of round 1's headline move." % (100 * ratio))
print("If that is large, the bracket is partly an artifact of the Euclidean metric and round 1's")
print("bracket claim must be restated as metric-relative. The SCORE and PURITY evidence is unaffected,")
print("because trees are invariant to this transform - which section 4 verifies rather than assumes.")

effect on e_1NN (negative = tighter bracket)
  information only (DERIVED -> D2)       : -0.0115
  METRIC only     (DERIVED -> DERIVED_log): +0.0296   <- adds ZERO information
  both            (DERIVED -> D2_log)     : +0.0246

round 1 for scale: BASE -> DERIVED was -0.0128 (0.3757 -> 0.3629), p = 9.3e-06

the pure-metric move is 231% the size of round 1's headline move.
If that is large, the bracket is partly an artifact of the Euclidean metric and round 1's
bracket claim must be restated as metric-relative. The SCORE and PURITY evidence is unaffected,
because trees are invariant to this transform - which section 4 verifies rather than assumes.


### 2.2 Neighbourhood purity, per family

Round 1's supporting evidence was that purity rose for **every** family, most for the ones the overlap
analysis found swallowed by the Exploits and Fuzzers attractors. Same measurement here, at k=10.

In [6]:
K = 10
pur, NBRC = {}, {}
for name, F in SPACES.items():
    Xa_tr, Xa_te = F.iloc[itr][att_tr], F.iloc[ite][att_te]
    sc = RobustScaler().fit(Xa_tr)
    nn = NearestNeighbors(n_neighbors=K, algorithm="brute", n_jobs=-1).fit(sc.transform(Xa_tr))
    _, ii = nn.kneighbors(sc.transform(Xa_te))
    NBRC[name] = fam_tr[ii]
    pur[name] = pd.Series((fam_tr[ii] == fam_te[:, None]).mean(1)).groupby(fam_te).mean()

P = pd.DataFrame(pur); P.index = [FAMN[i] for i in P.index]
P["support"] = pd.Series(fam_te).value_counts().sort_index().values
P["D2-DER"] = (P["D2"] - P["DERIVED"]).round(4)
P["log-DER"] = (P["DERIVED_log"] - P["DERIVED"]).round(4)
print("own-class neighbourhood purity at k=%d (test attack rows)" % K)
print(P.round(4).to_string())
print("\noverall: " + " | ".join("%s %.4f" % (k, pur[k].mean()) for k in SPACES))
print("round-1 reference: BASE 0.3570 -> DERIVED 0.3761")

own-class neighbourhood purity at k=10 (test attack rows)
                DERIVED      D2  DERIVED_log  D2_log  support  D2-DER  log-DER
Analysis         0.2987  0.3013       0.2143  0.2260       77  0.0026  -0.0844
Backdoor         0.2267  0.2367       0.2100  0.2111       90  0.0100  -0.0167
DoS              0.2204  0.2210       0.1869  0.1894      883  0.0006  -0.0335
Exploits         0.6797  0.6928       0.6626  0.6647     6182  0.0132  -0.0171
Fuzzers          0.6728  0.6873       0.6476  0.6498     5279  0.0145  -0.0252
Generic          0.4321  0.4516       0.4142  0.4125      899  0.0195  -0.0179
Reconnaissance   0.5665  0.5696       0.5374  0.5373     2406  0.0031  -0.0292
Shellcode        0.1403  0.1384       0.1270  0.1268      370 -0.0019  -0.0132
Worms            0.1479  0.1938       0.0792  0.0938       48  0.0458  -0.0687

overall: DERIVED 0.3761 | D2 0.3880 | DERIVED_log 0.3421 | D2_log 0.3457
round-1 reference: BASE 0.3570 -> DERIVED 0.3761


### 2.3 Did the targeted pairs actually move?

Round-2 columns were built for three specific pairs. This checks the intended mechanism directly:
what fraction of a family's neighbourhood belongs to the family it is confused with.

In [7]:
TARGET = [("Reconnaissance", "Exploits"), ("DoS", "Exploits"),
          ("Shellcode", "Fuzzers"), ("Analysis", "Exploits"),
          ("Worms", "Exploits"), ("Backdoor", "Exploits")]
rows = []
for a, b in TARGET:
    ia, ib = FAMN.index(a), FAMN.index(b)
    m = fam_te == ia
    r = dict(family=a, attracted_to=b, support=int(m.sum()))
    for name in SPACES:
        r[name] = round(float((NBRC[name][m] == ib).mean()), 4)
    r["D2-DER"] = round(r["D2"] - r["DERIVED"], 4)
    rows.append(r)
T = pd.DataFrame(rows)
print("share of a family's k=10 neighbourhood held by its attractor (lower = better separated)")
print(T.to_string(index=False))

share of a family's k=10 neighbourhood held by its attractor (lower = better separated)
        family attracted_to  support  DERIVED     D2  DERIVED_log  D2_log  D2-DER
Reconnaissance     Exploits     2406   0.1754 0.1731       0.1785  0.1798 -0.0023
           DoS     Exploits      883   0.4071 0.4145       0.4345  0.4345  0.0074
     Shellcode      Fuzzers      370   0.4619 0.4670       0.4424  0.4459  0.0051
      Analysis     Exploits       77   0.4805 0.4766       0.5494  0.5442 -0.0039
         Worms     Exploits       48   0.3896 0.3438       0.3792  0.3604 -0.0458
      Backdoor     Exploits       90   0.3089 0.3133       0.2811  0.2989  0.0044


## 3. End-to-end score, four arms matched

One gate + one $\alpha{=}0.25$ XGBoost Stage-B per arm — the same cheap matched comparison round 1
used to compare *representations* without confounding the feature change with combiner behaviour.

The two `_log` arms are the control: **a gradient-boosted tree is invariant to a strictly monotone
per-column transform**, so their scores should reproduce their un-logged twins to within
histogram-binning jitter. Measuring this rather than asserting it is the point — if the `_log` arms
scored differently, the invariance argument (and hence the whole metric-vs-information split above)
would be wrong.

In [8]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw); return XGBClassifier(**p)

wc_alpha = lambda n, a: ((n.sum() / n) ** a) / (((n.sum() / n) ** a).mean())
def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

res, PRED = [], {}
for name, F in SPACES.items():
    t0 = time.time()
    Xtr, Xte = F.iloc[itr], F.iloc[ite]
    ytr10, yte10 = y_arr[itr], y_arr[ite]

    gate = make_xgb(); yb = (ytr10 != 0).astype(int)
    gate.fit(Xtr, yb, sample_weight=compute_sample_weight("balanced", yb))
    gp = gate.predict(Xte)

    Xa, ya = Xtr[att_tr], fam_tr
    sb = make_xgb()
    sb.fit(Xa, ya, sample_weight=sample_w(ya, wc_alpha(np.bincount(ya, minlength=NC), 0.25)))

    yp = np.zeros(len(yte10), int); g = gp == 1
    yp[g] = np.vectorize(idx_to_fam.get, otypes=[int])(sb.predict(Xte[g]))
    PRED[name] = yp
    res.append(dict(space=name, features=F.shape[1],
                    stageB_mF1=round(f1_score(fam_te, sb.predict(Xte[att_te]),
                                              average="macro", zero_division=0), 4),
                    e2e_mF1=round(f1_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_mP=round(precision_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_mR=round(recall_score(yte10, yp, average="macro", zero_division=0), 4),
                    e2e_acc=round(float((yp == yte10).mean()), 4),
                    gate_P=round(precision_score((yte10 != 0).astype(int), gp), 4),
                    gate_R=round(recall_score((yte10 != 0).astype(int), gp), 4),
                    secs=round(time.time() - t0)))
    print("  %-12s done (%.0fs)" % (name, time.time() - t0))

R = pd.DataFrame(res)
print(); print(R.to_string(index=False))
b = R[R.space == "DERIVED"].iloc[0]
print("\nvs round-1 DERIVED (single-config yardstick +/-0.017; paired comparison band +/-0.012):")
for _, r in R.iterrows():
    if r["space"] != "DERIVED":
        print("  %-12s mF1 %+.4f | acc %+.4f | StageB %+.4f"
              % (r["space"], r["e2e_mF1"] - b["e2e_mF1"], r["e2e_acc"] - b["e2e_acc"],
                 r["stageB_mF1"] - b["stageB_mF1"]))
print("\npublished round-1 reference: BASE 0.6157/0.9081 -> DERIVED 0.6258/0.9117")

  DERIVED      done (35s)


  D2           done (57s)


  DERIVED_log  done (41s)


  D2_log       done (42s)

      space  features  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc  gate_P  gate_R  secs
    DERIVED        75      0.6090   0.6258  0.7031  0.5988   0.9117  0.9267  0.9961    35
         D2       113      0.6021   0.6185  0.6863  0.5920   0.9119  0.9293  0.9972    57
DERIVED_log        75      0.6020   0.6202  0.6904  0.5940   0.9118  0.9271  0.9960    41
     D2_log       113      0.6063   0.6227  0.6924  0.5941   0.9122  0.9293  0.9972    42

vs round-1 DERIVED (single-config yardstick +/-0.017; paired comparison band +/-0.012):
  D2           mF1 -0.0073 | acc +0.0002 | StageB -0.0069
  DERIVED_log  mF1 -0.0056 | acc +0.0001 | StageB -0.0070
  D2_log       mF1 -0.0031 | acc +0.0005 | StageB -0.0027

published round-1 reference: BASE 0.6157/0.9081 -> DERIVED 0.6258/0.9117


### 3.1 Control check: how invariant were the trees, exactly?

Prediction-level agreement between each arm and its logged twin. Anything below ~99% would mean the
histogram binning is doing something the invariance argument does not cover.

In [9]:
for a, b_ in (("DERIVED", "DERIVED_log"), ("D2", "D2_log")):
    ag = float((PRED[a] == PRED[b_]).mean())
    print("  %-8s vs %-12s : identical predictions on %.4f%% of test rows  (mF1 %.4f vs %.4f)"
          % (a, b_, 100 * ag,
             R[R.space == a].iloc[0]["e2e_mF1"], R[R.space == b_].iloc[0]["e2e_mF1"]))

  DERIVED  vs DERIVED_log  : identical predictions on 98.9908% of test rows  (mF1 0.6258 vs 0.6202)
  D2       vs D2_log       : identical predictions on 98.7899% of test rows  (mF1 0.6185 vs 0.6227)


### 3.2 Where the round-2 columns landed in the model

In [10]:
F = SPACES["D2"]
Xa, ya = F.iloc[itr][att_tr], fam_tr
sb = make_xgb(); sb.fit(Xa, ya, sample_weight=sample_w(ya, wc_alpha(np.bincount(ya, minlength=NC), 0.25)))
imp = pd.Series(sb.feature_importances_, index=F.columns).sort_values(ascending=False)
r1n, r2n = set(R1.columns), set(R2.columns)
tag = lambda k: "(r2)" if k in r2n else ("(r1)" if k in r1n else "")
print("top 25 features of the D2 Stage-B model")
for i, (k, v) in enumerate(imp.head(25).items()):
    print("  %2d. %-28s %.4f %s" % (i + 1, k, v, tag(k)))
tot = imp.sum()
print("\nimportance share: BASE %.1f%% | round-1 %.1f%% | round-2 %.1f%%"
      % (100 * imp[[c for c in imp.index if c not in r1n and c not in r2n]].sum() / tot,
         100 * imp[[c for c in imp.index if c in r1n]].sum() / tot,
         100 * imp[[c for c in imp.index if c in r2n]].sum() / tot))
print("column share : BASE %.1f%% | round-1 %.1f%% | round-2 %.1f%%"
      % (100 * (F.shape[1] - len(r1n) - len(r2n)) / F.shape[1],
         100 * len(r1n) / F.shape[1], 100 * len(r2n) / F.shape[1]))
print("\ntop 10 round-2 columns by importance:")
for i, (k, v) in enumerate(imp[[c for c in imp.index if c in r2n]].head(10).items()):
    print("  %2d. %-28s %.4f" % (i + 1, k, v))

top 25 features of the D2 Stage-B model
   1. handshake_bal                0.1452 (r2)
   2. frac_PSH                     0.0882 (r1)
   3. fwd_len_range_norm           0.0813 (r2)
   4. PSH Flag Count               0.0746 
   5. bwd_len_range_norm           0.0703 (r2)
   6. r_syn_fin                    0.0446 (r1)
   7. hdr_share                    0.0248 (r2)
   8. Bwd Packet Length Max        0.0245 
   9. Bwd Packet Length Min        0.0218 
  10. asym_byte                    0.0197 (r1)
  11. avg_pkt_size_rel             0.0194 (r2)
  12. payload_share_bwd            0.0133 (r2)
  13. Packet Length Max            0.0126 
  14. cv_pkt_len                   0.0125 (r1)
  15. Bwd Bytes/Bulk Avg           0.0112 
  16. hdr_per_pkt_bwd              0.0112 (r1)
  17. asym_pkt                     0.0111 (r1)
  18. Packet Length Mean           0.0106 
  19. Fwd Packet Length Max        0.0102 
  20. act_data_frac_fwd            0.0100 (r2)
  21. Packet Length Std            0.0089 
  22.

## 4. Does `D2` pay under the full v6 combiner?

Section 3 used a single $\alpha{=}0.25$ model, which is the right instrument for comparing
*representations*. But it is **not** sufficient to decide whether to adopt `D2`, because of what
`v6_significance.ipynb` established:

| Contrast | on BASE | on DERIVED |
|---|---|---|
| ensembling vs single model | +0.0029 (p = 0.555, **not** established) | **+0.0231 (p < 0.001)** |

The mechanism proposed there was that **the combiner only pays once neighbourhoods are pure enough**
for competence estimation to work — and purity is exactly what round 2 improved
(0.3761 -> 0.3880 overall, Worms +0.046).

That yields a directional prediction, stated before the measurement:

> If the purity mechanism is right, the *ensembling gain* should be **larger** on `D2` than on
> `DERIVED`, even though `D2`'s single-model score is lower.

This is a test of the project's own explanation, not a search for a better number: the combiner
configuration is **fixed in advance** at v6's `DYN OLA-cw k=30 p=8` (selected on SELV in
`final_model_v6.ipynb`), so nothing is being selected on test here. If the prediction fails, the
purity mechanism is weakened and `D2` is rejected on the evidence of both instruments.

In [11]:
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from imblearn.over_sampling import SMOTE

SPECS = ([(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS); I25 = MODELS.index("XGB a=0.25")

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate_full(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    m = make_xgb(); m.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb)); return m

def fuse_static(probas, Wc):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M): S += probas[m] * Wc[m][None, :]
    return S.argmax(1)

def fuse_dyn(probas, Wd, p=1.0):
    Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M): S += Wp[:, m:m+1] * probas[m]
    return S.argmax(1)

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s > best + 1e-7: best, cur = s, gg
                W[m_, c] = cur
    return W

prof = lambda ps: np.hstack(ps).astype(np.float32)
print("pool:", M, "members")

pool: 7 members


In [12]:
def v6_pipeline(space_name, F, cache):
    """final_model_v6.ipynb, verbatim in structure; only the feature space changes."""
    X_trf, X_test = F.iloc[itr].reset_index(drop=True), F.iloc[ite]
    y_trf, y_te = y_arr[itr], y_arr[ite]
    att = y_trf != 0
    fam = np.full(len(y_trf), -1); fam[att] = fam_tr

    OOF = None
    if cache.exists():
        z = np.load(cache)
        cand = [z[f"m{m}"] for m in range(M)]
        if cand[0].shape == (len(y_trf), NC):
            # alignment check: a correctly aligned OOF must classify attack rows far above chance
            acc = float((cand[I25][att].argmax(1) == fam[att]).mean())
            if acc > 0.5:
                OOF = cand
                print("  %-8s reusing %s (OOF alignment check: a=0.25 attack accuracy %.4f)"
                      % (space_name, cache.name, acc))
            else:
                print("  %-8s cache REJECTED - alignment check failed (%.4f)" % (space_name, acc))
    if OOF is None:
        OOF = [np.zeros((len(y_trf), NC), np.float32) for _ in range(M)]
        t0 = time.time()
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_trf):
            ma = y_trf[tr] != 0
            Xa = X_trf.iloc[tr][ma].reset_index(drop=True); ya = pd.Series(fam[tr][ma])
            for mi, (_, (k, p)) in enumerate(SPECS):
                OOF[mi][va] = fit_member(k, p, Xa, ya).predict_proba(X_trf.iloc[va]).astype(np.float32)
        np.savez_compressed(cache, **{f"m{m}": OOF[m] for m in range(M)})
        print("  %-8s cross-fit %.0fs -> %s" % (space_name, time.time() - t0, cache.name))

    comb_idx, _ = train_test_split(np.arange(len(y_trf)), test_size=0.5,
                                   random_state=RS + 1, stratify=y_trf)
    ref = comb_idx[att[comb_idx]]; y_ref = fam[ref]
    CORRECT = np.stack([(OOF[m][ref].argmax(1) == y_ref) for m in range(M)])
    WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)
    W_class = fit_W([OOF[m][ref] for m in range(M)], y_ref)
    NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
        prof([OOF[m][ref] for m in range(M)]))

    t0 = time.time()
    gate_f = fit_gate_full(X_trf, y_trf)
    Xa_f = X_trf[att].reset_index(drop=True); ya_f = pd.Series(fam[att])
    FINAL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
    gmask = gate_f.predict(X_test) == 1
    P = [m.predict_proba(X_test.iloc[np.where(gmask)[0]]).astype(np.float32) for m in FINAL]
    _, idx = NN_D.kneighbors(prof(P))
    print("  %-8s refit + test in %.0fs" % (space_name, time.time() - t0))

    def comp_ola(k):
        nb = idx[:, :k]; wn = WC_REF[y_ref[nb]]; den = wn.sum(1)
        Wd = np.empty((nb.shape[0], M))
        for m in range(M): Wd[:, m] = (CORRECT[m][nb] * wn).sum(1) / den
        return Wd

    out = []
    for tag, fam_g in (("single a=0.25", P[I25].argmax(1)),
                       ("static W", fuse_static(P, W_class)),
                       ("dyn k=30 p=8  (v6 config)", fuse_dyn(P, comp_ola(30), p=8)),
                       ("dyn k=100 p=8", fuse_dyn(P, comp_ola(100), p=8))):
        yp = np.zeros(len(y_te), int)
        yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_g)
        out.append(dict(space=space_name, config=tag,
                        mF1=round(f1_score(y_te, yp, average="macro", zero_division=0), 4),
                        acc=round(float((yp == y_te).mean()), 4),
                        f1_Worms=round(f1_score(y_te, yp, average=None, labels=[9],
                                                zero_division=0)[0], 3)))
    return pd.DataFrame(out)

E = pd.concat([v6_pipeline("DERIVED", SPACES["DERIVED"], Path("cache_oof_v6_derived.npz")),
               v6_pipeline("D2", SPACES["D2"], Path("cache_oof_d2.npz"))], ignore_index=True)
print(); print(E.to_string(index=False))

  DERIVED  reusing cache_oof_v6_derived.npz (OOF alignment check: a=0.25 attack accuracy 0.7412)


  DERIVED  refit + test in 167s


  D2       cross-fit 862s -> cache_oof_d2.npz


  D2       refit + test in 212s



  space                    config    mF1    acc  f1_Worms
DERIVED             single a=0.25 0.6258 0.9117     0.464
DERIVED                  static W 0.6459 0.9098     0.608
DERIVED dyn k=30 p=8  (v6 config) 0.6485 0.9123     0.582
DERIVED             dyn k=100 p=8 0.6466 0.9126     0.571
     D2             single a=0.25 0.6185 0.9119     0.459
     D2                  static W 0.6429 0.9127     0.523
     D2 dyn k=30 p=8  (v6 config) 0.6464 0.9135     0.568
     D2             dyn k=100 p=8 0.6445 0.9138     0.550


In [13]:
print("PRE-REGISTERED TEST: is the ensembling gain larger on D2 than on DERIVED?\n")
for sp in ("DERIVED", "D2"):
    s = E[(E.space == sp) & (E.config == "single a=0.25")].iloc[0]["mF1"]
    d = E[(E.space == sp) & (E.config == "dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"]
    w = E[(E.space == sp) & (E.config == "static W")].iloc[0]["mF1"]
    print("  %-8s single %.4f -> dyn %.4f   ensembling gain %+.4f   (static W %+.4f)"
          % (sp, s, d, d - s, w - s))
gd = (E[(E.space == "DERIVED") & (E.config == "dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"]
      - E[(E.space == "DERIVED") & (E.config == "single a=0.25")].iloc[0]["mF1"])
g2 = (E[(E.space == "D2") & (E.config == "dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"]
      - E[(E.space == "D2") & (E.config == "single a=0.25")].iloc[0]["mF1"])
print("\n  prediction (purity mechanism): gain(D2) > gain(DERIVED)")
print("  observed: %+.4f vs %+.4f  ->  %s" % (g2, gd, "SUPPORTED" if g2 > gd else "NOT SUPPORTED"))

best_d = E[E.space == "DERIVED"].sort_values("mF1", ascending=False).iloc[0]
best_2 = E[E.space == "D2"].sort_values("mF1", ascending=False).iloc[0]
print("\nADOPTION DECISION (paired comparison band +/-0.012)")
print("  v6 published (DERIVED, dyn k=30 p=8) : 0.6485 / 0.9123")
print("  DERIVED here, same config            : %.4f / %.4f" % (
      E[(E.space=="DERIVED") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"],
      E[(E.space=="DERIVED") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["acc"]))
print("  D2 here, same config                 : %.4f / %.4f" % (
      E[(E.space=="D2") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"],
      E[(E.space=="D2") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["acc"]))
delta = (E[(E.space=="D2") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"]
         - E[(E.space=="DERIVED") & (E.config=="dyn k=30 p=8  (v6 config)")].iloc[0]["mF1"])
print("  D2 - DERIVED at the fixed v6 config  : %+.4f  -> %s"
      % (delta, "ADOPT D2" if delta > 0.012 else
         ("REJECT D2 (worse beyond the band)" if delta < -0.012 else
          "NO MEASURABLE DIFFERENCE - keep DERIVED (fewer columns, already published)")))

PRE-REGISTERED TEST: is the ensembling gain larger on D2 than on DERIVED?

  DERIVED  single 0.6258 -> dyn 0.6485   ensembling gain +0.0227   (static W +0.0201)
  D2       single 0.6185 -> dyn 0.6464   ensembling gain +0.0279   (static W +0.0244)

  prediction (purity mechanism): gain(D2) > gain(DERIVED)
  observed: +0.0279 vs +0.0227  ->  SUPPORTED

ADOPTION DECISION (paired comparison band +/-0.012)
  v6 published (DERIVED, dyn k=30 p=8) : 0.6485 / 0.9123
  DERIVED here, same config            : 0.6485 / 0.9123
  D2 here, same config                 : 0.6464 / 0.9135
  D2 - DERIVED at the fixed v6 config  : -0.0021  -> NO MEASURABLE DIFFERENCE - keep DERIVED (fewer columns, already published)


## 5. Findings

**Round 2 is a negative result on the score, and it produced a methodological correction that matters
more than the score would have.** Two instruments that agreed in round 1 disagree here, and the
control identifies which one to stop trusting.

### 1. The headline: `D2` is rejected

| Space | features | $e_{1NN}$ | purity k=10 | single $\alpha{=}0.25$ | at the fixed v6 combiner |
|---|---|---|---|---|---|
| `DERIVED` (round 1) | 75 | 0.3632 | 0.3761 | **0.6258** | **0.6485** / 0.9123 |
| `D2` (round 2) | 113 | **0.3517** | **0.3880** | 0.6185 | 0.6464 / **0.9135** |

The geometric instruments both improve — the bracket tightens to $[0.1979, 0.3517]$ (McNemar
$p = 4.7\times10^{-7}$, net +186 rows) and purity rises in 8 of 9 families — while the score does
**not**: $-0.0073$ single-model, $-0.0021$ at the v6 configuration. Both are inside the $\pm 0.012$
paired band, so the correct statement is **no measurable difference**, not a loss. With no gain and 38
extra columns, `DERIVED` remains the feature set.

`DERIVED` reproduced the published v6 numbers to four decimals (0.6485 / 0.9123), so the comparison is
sound rather than a re-implementation artifact.

Note the direction of the small movement: `D2` is *slightly better on accuracy* at every combiner
setting (0.9135 vs 0.9123) and *slightly worse on macro-F1*, with Worms F1 falling 0.582 -> 0.568.
That is the large-class / rare-class trade this project has seen repeatedly, not an improvement.

### 2. THE MAIN RESULT — the Cover-Hart bracket is metric-dependent, and the metric effect is bigger than any information effect measured in this project

The `_log` arms apply $\operatorname{sign}(x)\log(1+|x|)$ per column. This adds **zero** information,
and the deployed model cannot see it: prediction-level agreement with the untransformed arm is
**98.99%** (`DERIVED`) and **98.79%** (`D2`), with score differences of 0.0056 and 0.0042 — histogram
binning jitter, as the invariance argument predicts.

Yet it moves the primary diagnostic more than any real feature change ever has:

| Change | $\Delta e_{1NN}$ | adds information? | p |
|---|---|---|---|
| round 1: `BASE` -> `DERIVED` (+31 columns) | $-0.0128$ | yes | 9.3e-06 |
| round 2: `DERIVED` -> `D2` (+38 columns) | $-0.0115$ | yes | 4.7e-07 |
| **control: `DERIVED` -> `DERIVED_log`** | **$+0.0296$** | **no** | **4.8e-22** |

**The pure-metric effect is 2.3x the size of round 1's headline result.**

What this does and does not overturn:

- **Overturned:** "the Cover-Hart ceiling moved" as a claim about the *problem*, or about what any
  classifier can achieve. $e_{1NN}$ is a property of *(features, metric)* jointly. A gradient-boosted
  tree is invariant to a transform that moves the bracket by twice the amount the new features did, so
  the bracket cannot bound it in the way the phrase implies. **The bracket must be quoted as
  metric-relative from now on.**
- **Not overturned:** the bracket is still a valid description of the geometry that *distance-based*
  methods actually operate in — the k-NN overlap analysis, neighbourhood purity, and the v5/v6
  DEC-space competence estimator all live in that geometry, and for them it is the right measurement.
- **Not overturned:** round 1's adoption decision. It rested on the score (+0.0101) and on purity as
  well as the bracket, and the score is metric-free. But round 1's argument that *"two independent
  instruments agreeing is stronger evidence than either alone"* does not survive round 2, where the
  same two instruments **disagree**. They were never independent measurements of the same thing.

A secondary observation worth one line in the write-up: log-compression made the k-NN geometry
**worse** (purity 0.3761 -> 0.3421, every family down). The heavy-tailed raw scale is more
discriminative under RobustScaler than a compressed one — the opposite of the usual intuition for
heavy-tailed features.

### 3. Why the round-2 columns are important to the model yet add nothing

They are used heavily: **43.9% of total importance on 33.6% of the columns**, and `handshake_bal` is
the single most important feature in the model (0.1452, ahead of `frac_PSH` and `PSH Flag Count`).

The explanation is visible in the definitions. `handshake_bal` $= (S-F)/(S+F)$ where $S$, $F$ are the
SYN and FIN counts; round 1 already contained `r_syn_fin` $= S/F$. Writing $r = S/F$,

$$\text{handshake\_bal} = \frac{r-1}{r+1}$$

which is **strictly monotone in `r_syn_fin`** — an exactly equivalent feature, differing only in being
bounded. A tree is indifferent between them (it simply splits on whichever it is given, and
`r_syn_fin` drops to 6th), while a Euclidean metric is not. The same holds for `fwd_len_range_norm`
and `bwd_len_range_norm` (#3 and #5), which are bounded re-expressions of round 1's
`fwd_len_span_rel` / `bwd_len_span_rel`.

**So the bulk of round 2 turned out to be re-encoding, not new information — which is precisely why
the bracket moved and the score did not.** This is the same mechanism as the `_log` control, arrived
at independently: three of the top five features are bounded monotone rewrites of columns that were
already present.

### 4. The targeted pairs mostly did not move

The columns were designed for specific confusion pairs. Measured as the share of a family's k=10
neighbourhood held by its attractor:

| Family -> attractor | DERIVED | D2 | |
|---|---|---|---|
| Reconnaissance -> Exploits | 0.1754 | 0.1731 | −0.002 |
| DoS -> Exploits | 0.4071 | 0.4145 | **+0.007 (worse)** |
| Shellcode -> Fuzzers | 0.4619 | 0.4670 | **+0.005 (worse)** |
| Analysis -> Exploits | 0.4805 | 0.4766 | −0.004 |
| **Worms -> Exploits** | 0.3896 | **0.3438** | **−0.046** |

Only Worms separated. The overall purity gain came mostly from the *large* families improving their
own neighbourhoods (Exploits +0.013, Fuzzers +0.015, Generic +0.020), not from the intended targets.
**The design hypothesis — that a scan's lack of response would separate Reconnaissance from
Exploits — failed on its own terms**, and should be reported that way rather than absorbed into the
overall purity number.

### 5. Pre-registered mechanism test: supported

`v6_significance.ipynb` explained v6 as an interaction — the combiner pays only once neighbourhoods
are pure enough. `D2` is the purer space, so the prediction was a *larger ensembling gain* on `D2`
even though its single-model score is lower.

| Space | single | v6 combiner | ensembling gain |
|---|---|---|---|
| `DERIVED` | 0.6258 | 0.6485 | +0.0227 |
| `D2` | 0.6185 | 0.6464 | **+0.0279** |

Observed as predicted. The mechanism now has support from a second, independent feature space — but
the difference of differences is +0.005, itself inside the noise band, so this is **corroborating
evidence for the mechanism, not an established effect**.

### 6. Consequence: algebra on these 76 columns is exhausted

Two rounds of derived features: round 1 marginal (+0.0101, $p = 0.071$, not established alone),
round 2 null. The remaining directions named in `NEXT_STEPS.md` are not merely expensive but
**impossible on this file** — verified in section 1, `CIC-UNSW-Data.csv` contains no IP, port or
timestamp column, so per-host and per-window aggregates cannot be constructed, and percentile flow
statistics would require re-running CICFlowMeter on PCAPs the project does not have.

**"More features" should therefore be closed as a score lever for this dataset**, and the plateau
restated in the thesis as a limit of *what CICFlowMeter records*, which is a stronger and more
defensible claim than a limit of the 44-column pruned subset. The remaining open items are
cross-dataset evaluation and SHAP, neither of which is about the score.